# Lab 8 - Sohum Pohane

In [ ]:
from google.colab import drive
gdrive_mount = '/content/drive'
drive.mount(gdrive_mount, force_remount=True)

Mounted at /content/drive


In [ ]:
labdir = '/content/drive/My Drive/Teaching/605.646/Fall 2025/lab8/'
%cd "$labdir"

/content/drive/My Drive/Teaching/605.646/Fall 2025/lab8


In [ ]:
import spacy
import re
import json
import pandas as pd
import os
from collections import Counter

In [ ]:
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

In [ ]:
def parse_obits(filename):
  with open(filename, 'r', encoding='utf-8') as f:
    content = f.read()
  raw_docs = content.split('</P>')
  parsed_data = []
  for doc in raw_docs:
    if '<P' in doc:
      id_match=re.search(r'ID=(\d+)>', doc)
      doc_id=int(id_match.group(1)) if id_match else 0
      clean_text=re.sub(r'<P.*?>', '', doc).strip()
      clean_text=clean_text.replace('\n', ' ')
      parsed_data.append({"id": doc_id, "text": clean_text})

  return parsed_data

In [ ]:
train_docs = parse_obits("obits.train.txt")
print(f"Loaded {len(train_docs)} documents.")

Loaded 20 documents.


In [ ]:
test_docs = parse_obits("obits.test.txt")
print(f"Loaded {len(test_docs)} documents.")

Loaded 10 documents.


In [ ]:
import re
from collections import Counter
occupations_list = ["doctor", "physician", "nurse", "teacher", "professor", "engineer","veteran", "soldier", "pilot", "carpenter", "builder", "manufacturer","business owner", "manager"]
male_tokens= ["he","him","his","husband","father","son","brother","grandfather"]
female_tokens= ["she","her","hers","wife","mother","daughter","sister","grandmother"]

def extract_name(doc_obj):
    text = doc_obj.text
    pattern =r'^([A-Z][\w\s“”.()]+?)(?:,|\sdied|\spassed)'
    match =re.search(pattern, text)
    if match:
        return match.group(1).strip()
    return "Unknown"
def extract_sex(doc_obj):
    counts =Counter([t.text.lower() for t in doc_obj])
    m_score =sum(counts[t] for t in male_tokens)
    f_score =sum(counts[t] for t in female_tokens)
    return "Male" if m_score > f_score else "Female"

def extract_age(text):
    pattern =r'(?:,\s|aged\s)(\d{2,3})(?:,|\syears)'
    match =re.search(pattern, text[:150])
    if match:
        return int(match.group(1))
    return None

def extract_locations(doc_obj):
    locs =set()
    for i, sent in enumerate(doc_obj.sents):
        if i > 2:
            break
        for ent in sent.ents:
            if ent.label_ == "GPE":
                locs.add(ent.text)
    return list(locs)

def extract_spouse(doc_obj):
    text =doc_obj.text
    p1 = r'(?:wife|husband)\s+(?:of\s+[\w\s]+\s+years,?)?\s+([A-Z][a-z]+)'
    m1 = re.search(p1, text)
    if m1:
        return m1.group(1)
    p2 = r'married\s+([A-Z][a-z]+)'
    m2 = re.search(p2, text)
    if m2:
        return m2.group(1)
    return None

def extract_occupation(doc_obj):
    occupations = []

    for token in doc_obj:
        if token.lemma_.lower() in occupations_list:
            occupations.append(token.text)

    for ent in doc_obj.ents:
        if ent.label_== "ORG":
            start = max(0, ent.start - 3)
            prev_text = doc_obj[start:ent.start].text.lower()
            if "worked for" in prev_text or "employed by" in prev_text:
                occupations.append(f"Employee at {ent.text}")

    return list(set(occupations))

def extract_birthdate(text):
    born_year= re.search(r"born.*?(\d{4})", text, re.IGNORECASE)
    if born_year:
        return born_year.group(1)
    return None

def process(raw_text, nlp):
    doc =nlp(raw_text)
    return {"name": extract_name(doc),"sex": extract_sex(doc),"age": extract_age(raw_text),
        "locations": extract_locations(doc),"spouse": extract_spouse(doc),"birthdate": extract_birthdate(raw_text),
        "occupation": extract_occupation(doc)}


In [ ]:
trad_results = {}
for doc in test_docs:
    trad_results[doc['id']] = process(doc['text'],nlp)
trad_results

{100: {'name': 'Bernard Pugh',
  'sex': 'Male',
  'age': 90,
  'locations': ['Earl and Beryl', 'Davis County'],
  'spouse': None,
  'birthdate': '1928',
  'occupation': ['teacher']},
 101: {'name': 'Sherri Oden',
  'sex': 'Female',
  'age': 62,
  'locations': ['Cincinnati', 'Missouri', 'Des Moines', 'Unionville'],
  'spouse': 'Paul',
  'birthdate': '1956',
  'occupation': []},
 102: {'name': 'Dennis Strickler',
  'sex': 'Male',
  'age': None,
  'locations': ['Des Moines', 'E.J.', 'Moravia', 'Iowa', 'Centerville'],
  'spouse': None,
  'birthdate': '1957',
  'occupation': []},
 103: {'name': 'Candita Marie Furlin',
  'sex': 'Female',
  'age': None,
  'locations': ['Centerville', 'Des Moines', 'Iowa', 'Seymour'],
  'spouse': None,
  'birthdate': '1976',
  'occupation': []},
 104: {'name': 'Virginia Elizabeth Koestner',
  'sex': 'Female',
  'age': None,
  'locations': ['Virginia', 'Moravia'],
  'spouse': None,
  'birthdate': '1921',
  'occupation': []},
 105: {'name': 'Florence R. Schau',


In [ ]:
def get_llm_extraction(text):
  from openai import OpenAI
  import json
  client = OpenAI(api_key="YOUR_OPENAI_API_KEY")
  prompt = f"Extract the following fields from the text into a valid JSON object: name, sex, age, locations (list), spouse, birthdate, occupation (list). Return ONLY the JSON object. Do not use markdown formatting and only give it in one line.\n\nText: {text}"
  try:
      response = client.chat.completions.create(model="gpt-3.5-turbo",messages=[{"role":"user", "content": prompt}],temperature=0)
      content = response.choices[0].message.content.strip()
      return json.loads(content.strip())

  except Exception as e:
      return "NOT FOUND"

In [ ]:
llm_results = {}
for doc in test_docs:
    llm_results[doc['id']] = get_llm_extraction(doc['text'])


In [ ]:
llm_results

{100: {'name': 'Bernard Pugh',
  'sex': 'male',
  'age': 90,
  'locations': ['Centerville',
   'Fox River Township',
   'Davis County',
   'Unionville',
   'Iowa',
   'Watertown',
   'South Dakota'],
  'spouse': 'Jewell Richmond',
  'birthdate': 'December 22, 1928',
  'occupation': ['high school teacher',
   'guidance counselor',
   'school superintendent',
   'financial advisor',
   'cattle farmer']},
 101: {'name': 'Sherri Oden',
  'sex': 'female',
  'age': 62,
  'locations': ['Cincinnati', 'Des Moines', 'Unionville', 'Centerville'],
  'spouse': 'Paul Oden',
  'birthdate': 'July 31, 1956',
  'occupation': ['dietary supervisor', 'health and nutrition']},
 102: {'name': 'Dennis Strickler',
  'sex': 'male',
  'age': 62,
  'locations': ['Moravia',
   'Des Moines',
   'Centerville',
   'Moravia Grace United Methodist Church',
   'Eddyville',
   'Albia',
   'Ankeny',
   'Blakesburg',
   'Ottumwa',
   'Newton',
   'Iconium Cemetery'],
  'spouse': 'Vicky Lockman',
  'birthdate': 'February 28

In [ ]:
gt_dict = {
  100: {
      "name": "Bernard Pugh",
      "sex": "Male",
      "age": 90,
      "locations": ["Centerville"],
      "spouse": "Jewell Richmond",
      "birthdate": "December 22, 1928",
      "occupation": ["High school teacher","Guidance counselor","School superintendent","Financial advisor","Cattle farmer","Served in the Signal Corps"]
  },
  101: {
      "name": "Sherri Oden",
      "sex": "Female",
      "age": 62,
      "locations": ["Cincinnati"],
      "spouse": "Paul Oden",
      "birthdate": "July 31, 1956",
      "occupation": ["Dietary supervisor"]
  },
  102: {
      "name": "Dennis Strickler",
      "sex": "Male",
      "age": 62,
      "locations": ["Moravia"],
      "spouse": "Vicky Lockman",
      "birthdate": "February 28, 1957",
      "occupation": ["Electrician"]
  },
  103: {
      "name": "Candita Marie Furlin",
      "sex": "Female",
      "age": 42,
      "locations": ["Des Moines"],
      "spouse": "Fiancé: Matt Horak",
      "birthdate": "August 3, 1976",
      "occupation": []
  },
  104: {
      "name": "Virginia Elizabeth Koestner",
      "sex": "Female",
      "age": 97,
      "locations": ["Centerville"],
      "spouse": "John Koestner",
      "birthdate": "November 2, 1921",
      "occupation": ["Worked within family businesses"]
  },
  105: {
      "name": "Florence R. Schau",
      "sex": "Female",
      "age": 91,
      "locations": ["South Wheeling"],
      "spouse": "Tony Schau",
      "birthdate": "December 6, 1927",
      "occupation": ["Member of Trinity Lutheran Church","Part owner of the Double J Club","Homemaker"]
  },
  106: {
      "name": "Linda Kay Yee",
      "sex": "Female",
      "age": 68,
      "locations": ["Wheeling"],
      "spouse": None,
      "birthdate": "February 1, 1951",
      "occupation": []
  },
  107: {
      "name": "Clyde William “Casey” Caseman",
      "sex": "Male",
      "age": 90,
      "locations": ["Wheeling"],
      "spouse": "Alicia McCombs Caseman",
      "birthdate": "February 4, 1929",
      "occupation": ["Served in the Army during the Korean War"]
  },
  108: {
      "name": "John Patrick “Pud” Moyle",
      "sex": "Male",
      "age": 97,
      "locations": ["Wheeling"],
      "spouse": "Mary Jane (Corcoran) Moyle",
      "birthdate": "September 8, 1921",
      "occupation": ["Tour guide","Worked the Jamboree In The Hills information booth"]
  },
  109: {
      "name": "William Spencer Haines",
      "sex": "Male",
      "age": 103,
      "locations": ["Weirton"],
      "spouse": "Ruth Kiger Haines",
      "birthdate": "February 9, 1916",
      "occupation": ["Insurance Agent","Served in the Army Medical Department"]
  }
}


In [ ]:
fields = ["name","sex","age","locations","spouse","occupation","birthdate"]

In [ ]:
for field in fields:
  print(field)
  for id in range(100,110):
    print(id)
    print(f"ground truth: {gt_dict[id][field]}")
    print(f"traditional: {trad_results[id][field]}")
    print(f"llm: {llm_results[id][field]}")
  print("\n")

name
100
ground truth: Bernard Pugh
traditional: Bernard Pugh
llm: Bernard Pugh
101
ground truth: Sherri Oden
traditional: Sherri Oden
llm: Sherri Oden
102
ground truth: Dennis Strickler
traditional: Dennis Strickler
llm: Dennis Strickler
103
ground truth: Candita Marie Furlin
traditional: Candita Marie Furlin
llm: Candita Marie Furlin
104
ground truth: Virginia Elizabeth Koestner
traditional: Virginia Elizabeth Koestner
llm: Virginia Elizabeth Koestner
105
ground truth: Florence R. Schau
traditional: Florence R. Schau
llm: Florence R. Schau
106
ground truth: Linda Kay Yee
traditional: Linda Kay Yee
llm: Linda Kay Yee
107
ground truth: Clyde William “Casey” Caseman
traditional: Clyde William “Casey” Caseman
llm: Clyde William Caseman
108
ground truth: John Patrick “Pud” Moyle
traditional: John Patrick “Pud” Moyle
llm: John Patrick "Pud" Moyle
109
ground truth: William Spencer Haines
traditional: Haines
llm: William Spencer Haines


sex
100
ground truth: Male
traditional: Male
llm: male

###Field: name
Ground Truth Total: 10

Traditional:

Analysis: IDs 100–108 are exact matches. ID 109 is a generous match.

TP: 10 | Sys: 10

LLM:

Analysis: All IDs match (minor differences like "Casey" or "Pud" ignored under generous scoring).

TP: 10 | Sys: 10

###Field: sex
Ground Truth Total: 10.

Traditional: All match

TP: 10 | Sys: 10

LLM: All match

TP: 10 | Sys: 10

###Field: age
Ground Truth Total: 10.

Traditional:

Hits: 6.

Misses: 4

TP: 6 | Sys: 6

LLM: All match

TP: 10 | Sys: 10

###Field: locations
Ground Truth Total: 10

Traditional:

Hits: TP=7

Total Output (Sys): 2+4+5+4+2+2+1+2+2+5 = 29.

LLM:

Hits (TP=10): TP=10

Total Output (Sys): 7+4+11+3+1+4+8+4+1+11 = 54.

Note: Precision suffers here because the LLM extracted every mentioned city (birthplace, children's homes), not just the residence.

###Field: spouse
Ground Truth Total: 9 (ID 106 is "None").

Traditional:

Hits: TP = 3

Sys: 3

LLM:

Hits: TP = 9

Sys: 9.

###Field: occupation
Ground Truth Total: 17

Traditional:

Hits: TP = 3

Sys: 3

LLM:

Hits: TP = 14

Sys: 21


###Field: birthdate
Ground Truth Total: 10

Traditional Method:

Output: The traditional method consistently extracted only the Year which i'll use a generous match

TP: 10 | Sys: 10

LLM Method:

Output: The LLM extracted the exact full date for every entry.

TP: 10 | Sys: 10

#Final Results
## <u> Name </u>

###Traditional
Precision: 10 / 10 (100%)

Recall: 10 / 10 (100%)

F1-Score: 100%

###LLM

Precision: 10 / 10 (100%)

Recall: 10 / 10 (100%)

F1-Score: 100%

##<u>Sex</u>

###Traditional
Precision: 10 / 10 (100%)

Recall: 10 / 10 (100%)

F1-Score: 100%

###LLM

Precision: 10 / 10 (100%)

Recall: 10 / 10 (100%)

F1-Score: 100%

##<u>Age</u>

###Traditional
Precision: 6 / 6 (100%)

Recall: 6 / 10 (60%)

F1-Score: 75%

###LLM

Precision: 10 / 10 (100%)

Recall: 10 / 10 (100%)

F1-Score: 100%

##<u>Locations</u>

###Traditional
Precision: 7 / 29 (24.1%)

Recall: 7 / 10 (70%)

F1-Score: 35.9%

###LLM

Precision: 10 / 54 (18.5%)

Recall: 10 / 10 (100%)

F1-Score: 31.3%

##<u>Spouse</u>

###Traditional
Precision: 3 / 3 (100%)

Recall: 3 / 9 (33.3%)

F1-Score: 50%

###LLM

Precision: 9 / 9 (100%)

Recall: 9 / 9 (100%)

F1-Score: 100%

##<u>Occupation</u>

###Traditional
Precision: 3 / 3 (100%)

Recall: 3 / 17 (17.6%)

F1-Score: 29.9%

###LLM

Precision: 14 / 21 (66.7%)

Recall: 14 / 17 (82.4%)

F1-Score: 73.7%

##<u>Birthdate</u>

###Traditional
Precision: 10 / 10 (100%)

Recall: 10 / 10 (100%)

F1-Score: 100%

###LLM

Precision: 10 / 10 (1000%)

Recall: 10 / 10 (100%)

F1-Score: 100%

We see that occupations are hard to identify, 'Occupation' relies on a list. For things that don't appear in the small list, it won't recognize it as an occupation. Locations are also hard to identify, traditional method, NER, picks up all of the locations (False Positives). It's easy to identify names, sex, and birthdate. However, it's important to note the way I get birthdate with traditional is just getting the year, and not the whole birthdate, I treated this as still a match.